# Lección 2: Técnicas de clustering
## Curso de Invierno — Microbioma de la rizósfera
### Instructor: David Alberto García-Estrada

⏱ **Duración estimada: ~90 min**

---

## Objetivos de aprendizaje
Al completar esta lección, serás capaz de:
1. Calcular e interpretar una matriz de distancias de Bray-Curtis entre comunidades microbianas.
2. Visualizar la estructura de las muestras con una ordenación (PCoA).
3. Aplicar clustering jerárquico y k-means, y evaluarlos con la silhouette.
4. Contrastar los clusters con variables del metadato y distinguir **estructura de los datos** de **efecto biológico**.

---

## 🔧 Setup y carga de datos (~5 min)

In [ ]:
# Detección de entorno (VSCode local vs Google Colab) y preparación de datos
import sys, zipfile, urllib.request
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
LESSON = 'leccion_1'   # ← en la Lección 2 cambiar a 'leccion_2'

# Subconjunto del taller en el repositorio (rama main)
SUBSET_URL = ('https://github.com/MicroAgroBiome/MicroAgroBiome_bioinfo2026_libro/raw/main/datos/subset.zip')

if IN_COLAB:
    !pip install -q "scikit-bio>=0.6.2" biom-format
    BASE = Path('/content')
else:
    # Local: sube hasta encontrar la carpeta datos/ (repo clonado); si no existe, usa la carpeta actual
    BASE = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'datos').is_dir()), Path.cwd())

DATA_PATH = BASE / 'datos'
SUBSET_PATH = DATA_PATH / 'subset'
RESULTS_PATH = BASE / 'resultados' / LESSON

# Si los datos no están descomprimidos: usa el zip local o lo descarga del repo
if not (SUBSET_PATH / 'taller_genus.biom').exists():
    zip_path = DATA_PATH / 'subset.zip'
    if not zip_path.exists():
        DATA_PATH.mkdir(parents=True, exist_ok=True)
        print("Descargando datos del taller...")
        urllib.request.urlretrieve(SUBSET_URL, zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(DATA_PATH)

# Rutas como texto terminadas en '/', igual que antes: el resto del notebook no cambia
DATA_PATH, SUBSET_PATH, RESULTS_PATH = (str(p) + '/' for p in (DATA_PATH, SUBSET_PATH, RESULTS_PATH))

print(f"Entorno: {'Google Colab' if IN_COLAB else 'Local (VSCode)'}")
print(f"SUBSET_PATH:  {SUBSET_PATH}")
print(f"RESULTS_PATH: {RESULTS_PATH}")

In [ ]:
%matplotlib inline

import warnings
import numpy as np
import pandas as pd
import biom
import skbio
from skbio import DistanceMatrix
from skbio.stats.distance import permanova
from skbio.stats.ordination import pcoa
import scipy
from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster, cophenet
import sklearn
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples, adjusted_rand_score
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", message=".*negative eigenvalues.*")  # aviso esperado en PCoA con Bray-Curtis

for lib in (pd, np, biom, skbio, scipy, sklearn, matplotlib, sns):
    print(f"{lib.__name__}: {lib.__version__}")

Cargamos el mismo subconjunto de la Lección 1 y repetimos su preparación (cada notebook es independiente): nos quedamos con **Bacteria y Archaea** y calculamos **abundancias relativas** (proporción de las lecturas de cada muestra), que es lo que se compara con Bray-Curtis.

In [ ]:
biom_table = biom.load_table(SUBSET_PATH + "taller_genus.biom")
abundance_table = biom_table.to_dataframe(dense=True).T.round().astype("int64")
metadata_df = pd.read_csv(SUBSET_PATH + "taller_metadata.csv", index_col=0)
assert list(abundance_table.index) == list(metadata_df.index)

# Dominio de cada género -> nos quedamos con Bacteria y Archaea
domain = pd.Series({g: m["taxonomy"][0].split("__", 1)[1]
                    for g, m in zip(biom_table.ids(axis="observation"), biom_table.metadata(axis="observation"))})
abundance_clean = abundance_table.loc[:, domain.reindex(abundance_table.columns).isin(["Bacteria", "Archaea"]).values]

# Abundancia relativa: cada muestra suma 1
rel_abundance = abundance_clean.div(abundance_clean.sum(axis=1), axis=0)

# Identificador corto de estudio (igual que en la Lección 1)
studies = sorted(metadata_df["doi"].unique())
metadata_df["study_id"] = metadata_df["doi"].map({doi: f"E{i + 1:02d}" for i, doi in enumerate(studies)})

print(f"rel_abundance: {rel_abundance.shape[0]} muestras x {rel_abundance.shape[1]} géneros")
print(f"Cada muestra suma 1: {np.allclose(rel_abundance.sum(axis=1), 1)}")

---

## 📖 Sección 1: Distancia entre comunidades (~10 min)

Para agrupar muestras necesitamos una noción de **qué tan parecidas** son dos comunidades. La **distancia de Bray-Curtis** compara las abundancias relativas género por género:

- **0** → comunidades idénticas.
- **1** → comunidades sin ningún género en común.

Pesa más a los géneros abundantes, así que no la distorsionan los géneros muy raros. Calculamos la distancia entre **todos los pares** de muestras y la guardamos en una **matriz de distancias** (simétrica, con ceros en la diagonal):

In [ ]:
dist_array = squareform(pdist(rel_abundance.values, metric="braycurtis"))
dist_df = pd.DataFrame(dist_array, index=rel_abundance.index, columns=rel_abundance.index)

print(f"Matriz de distancias: {dist_df.shape}")
dist_df.iloc[:5, :5].round(3)

In [ ]:
#| label: fig-l2-distancias
#| fig-cap: "Distribución de las distancias de Bray-Curtis entre todos los pares de muestras."
# Distribución de las distancias entre todos los pares distintos de muestras
pair_distances = dist_array[np.triu_indices_from(dist_array, k=1)]

fig, ax = plt.subplots(figsize=(6, 4))
sns.histplot(pair_distances, bins=40, ax=ax)
ax.set_xlabel("Distancia de Bray-Curtis")
ax.set_ylabel("Número de pares de muestras")
ax.set_title("Distancias entre pares de muestras")
plt.show()
print(f"Distancia mediana: {np.median(pair_distances):.2f} | mínima: {pair_distances.min():.2f} | máxima: {pair_distances.max():.2f}")

### ✏️ Ejercicio 1: ¿Importa la métrica?

**Instrucción:** calcula la matriz de distancias con la métrica **euclidiana** (`"euclidean"`) y mira qué tan bien coincide con Bray-Curtis. Tiempo estimado: 5 min.

In [ ]:
# ### TU CÓDIGO AQUÍ ###
euclid_array = squareform(pdist(rel_abundance.values, metric=TU_CODIGO_AQUI))

euclid_pairs = euclid_array[np.triu_indices_from(euclid_array, k=1)]
print(f"Correlación entre distancias Bray-Curtis y euclidianas: {np.corrcoef(pair_distances, euclid_pairs)[0, 1]:.2f}")

<details>
<summary>Ver solución</summary>

```python
euclid_array = squareform(pdist(rel_abundance.values, metric="euclidean"))
```

La euclidiana da más peso a los géneros muy abundantes y trata distinto la diferencia entre abundancias; por eso en ecología microbiana se prefieren métricas como Bray-Curtis. Correlación alta ≠ resultados idénticos.
</details>

---

## 📖 Sección 2: Ver la estructura — Ordenación PCoA (~10 min)

Una matriz de 75×75 números no se puede "mirar". El **PCoA** (*Principal Coordinates Analysis*) resume la matriz de distancias en pocos **ejes**: cada muestra pasa a ser un punto en un plano, de modo que **puntos cercanos = comunidades parecidas**. El primer eje (PC1) captura la mayor parte de la variación, el segundo (PC2) la siguiente, etc.

In [ ]:
dm = DistanceMatrix(dist_array, ids=list(rel_abundance.index))
pcoa_result = pcoa(dm, number_of_dimensions=5)

coords = pcoa_result.samples                      # coordenadas de cada muestra (PC1..PC5)
explained = pcoa_result.proportion_explained      # fracción de la variación que captura cada eje
print("Variación explicada por PC1..PC5:", (explained.iloc[:5] * 100).round(1).tolist(), "%")

In [ ]:
#| label: fig-l2-pcoa
#| fig-cap: "PCoA sobre Bray-Curtis: las mismas muestras coloreadas por familia de planta (izq.) y por estudio (der.)."
plot_df = coords[["PC1", "PC2"]].join(metadata_df[["host_name_family", "study_id"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="host_name_family", s=70, ax=axes[0])
axes[0].set_title("Coloreado por familia de la planta")
axes[0].legend(title="Familia")
sns.scatterplot(data=plot_df, x="PC1", y="PC2", hue="study_id", palette="tab20", s=70, ax=axes[1])
axes[1].set_title("Coloreado por estudio")
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", ncol=2, fontsize=8, title="Estudio")
for ax in axes:
    ax.set_xlabel(f"PC1 ({explained.iloc[0] * 100:.1f} %)")
    ax.set_ylabel(f"PC2 ({explained.iloc[1] * 100:.1f} %)")
plt.tight_layout()
plt.show()

**Observa:** los mismos puntos, dos colores. Hay unos pocos puntos **muy alejados** del resto: ¿qué serán? Y ¿en cuál de los dos paneles se ven *grupos* más claros? Guarda esas impresiones; las pondremos a prueba en la Sección 5.

---

## 📖 Sección 3: Clustering jerárquico y dendrograma (~15 min)

El **clustering jerárquico aglomerativo** empieza con cada muestra como su propio grupo y va **uniendo los dos grupos más cercanos**, paso a paso, hasta que queda uno solo. El resultado es un **dendrograma**. El **método de enlace** (*linkage*) define cómo se mide la distancia entre grupos; `average` usa la distancia promedio entre todos los pares de muestras de los dos grupos.

In [ ]:
condensed = squareform(dist_array, checks=False)          # formato que pide linkage()
linkage_avg = linkage(condensed, method="average")

In [ ]:
#| label: fig-l2-dendrograma
#| fig-cap: "Dendrograma con enlace promedio (etiqueta = estudio; color = familia de planta)."
family_colors = dict(zip(sorted(metadata_df["host_name_family"].unique()), sns.color_palette("Set1", 3)))

fig, ax = plt.subplots(figsize=(15, 5))
tree = dendrogram(linkage_avg, labels=metadata_df["study_id"].tolist(), leaf_rotation=90, leaf_font_size=9,
                  color_threshold=0, above_threshold_color="gray", ax=ax)

# Etiqueta = estudio; color de la etiqueta = familia de la planta
for tick, leaf in zip(ax.get_xmajorticklabels(), tree["leaves"]):
    tick.set_color(family_colors[metadata_df["host_name_family"].iloc[leaf]])

ax.set_ylabel("Distancia de Bray-Curtis")
ax.set_title("Dendrograma (enlace promedio)")
ax.grid(False)
handles = [plt.Line2D([0], [0], marker="s", linestyle="", color=c, label=f) for f, c in family_colors.items()]
ax.legend(handles=handles, title="Familia", bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

El mismo árbol, ahora junto a la **matriz de distancias** reordenada según el dendrograma: los bloques oscuros son grupos de muestras muy parecidas. Las franjas de color indican la familia de la planta y el estudio de cada muestra:

In [ ]:
#| label: fig-l2-clustermap
#| fig-cap: "Matriz de distancias reordenada según el dendrograma, con franjas de familia y estudio."
study_palette = dict(zip(sorted(metadata_df["study_id"].unique()), sns.color_palette("tab20", metadata_df["study_id"].nunique())))
side_colors = pd.DataFrame({"Familia": metadata_df["host_name_family"].map(family_colors),
                            "Estudio": metadata_df["study_id"].map(study_palette)})

g = sns.clustermap(dist_df, row_linkage=linkage_avg, col_linkage=linkage_avg,
                   row_colors=side_colors, col_colors=side_colors, cmap="viridis_r",
                   xticklabels=False, yticklabels=False, figsize=(8, 8), cbar_kws={"label": "Bray-Curtis"})
g.fig.suptitle("Distancias entre muestras (ordenadas por el dendrograma)", y=1.01)
plt.show()

Para obtener **clusters** "cortamos" el árbol: pedimos un número de grupos con `fcluster`. Revisa siempre **cuántas muestras** cae en cada uno:

In [ ]:
hier_labels = fcluster(linkage_avg, t=3, criterion="maxclust")
print("Muestras por cluster (jerárquico, 3 grupos):")
print(pd.Series(hier_labels).value_counts().sort_index().to_string())

coph_avg, _ = cophenet(linkage_avg, condensed)
print(f"\nCorrelación cofenética (enlace promedio): {coph_avg:.3f}")

La **correlación cofenética** mide qué tan fielmente el dendrograma conserva las distancias originales (más cerca de 1 = mejor).

### ✏️ Ejercicio 2: Cambiar el método de enlace

**Instrucción:** prueba el enlace **`"complete"`** (distancia entre los pares *más lejanos*) y compara su correlación cofenética con la del enlace promedio. Tiempo estimado: 5 min.

In [ ]:
# ### TU CÓDIGO AQUÍ ###
method = "TU_CODIGO_AQUI"     # prueba "complete"

linkage_other = linkage(condensed, method=method)
coph_other, _ = cophenet(linkage_other, condensed)
sizes = pd.Series(fcluster(linkage_other, t=3, criterion="maxclust")).value_counts().sort_index()

print(f"Cofenética: promedio = {coph_avg:.3f} | {method} = {coph_other:.3f}")
print(f"Muestras por cluster con {method}: {sizes.tolist()}")

<details>
<summary>Ver solución</summary>

```python
method = "complete"
```

Cofenética: promedio 0.989 vs complete 0.967 → el enlace promedio conserva mejor las distancias originales. Con 3 grupos ambos separan los mismos casos aislados (tamaños [3, 3, 69]).
</details>

---

## 📖 Sección 4: k-means y silhouette (~15 min)

**k-means** reparte las muestras en *k* grupos buscando que cada muestra quede cerca del centro de su grupo. Hay que **elegir k de antemano**, y trabaja con coordenadas (no con una matriz de distancias): usamos las **5 primeras coordenadas del PCoA**.

¿Cómo elegir *k*? La **silhouette** mide, para cada muestra, qué tan cerca está de su propio grupo frente al grupo vecino: va de −1 a 1; **más alta = grupos más separados**. Probamos varios *k* con una **semilla fija** (`random_state`) para que el resultado sea reproducible:

In [ ]:
#| label: fig-l2-eleccion-k
#| fig-cap: "Silhouette promedio de k-means para k = 2 a 8."
pcs = coords.values   # 75 muestras x 5 coordenadas

silhouette_by_k = {}
for k in range(2, 9):
    labels = KMeans(n_clusters=k, n_init=20, random_state=42).fit_predict(pcs)
    silhouette_by_k[k] = silhouette_score(pcs, labels)

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.plot(list(silhouette_by_k), list(silhouette_by_k.values()), marker="o")
ax.set_xlabel("Número de clusters (k)")
ax.set_ylabel("Silhouette promedio")
ax.set_title("Elección de k")
plt.show()

best_k = max(silhouette_by_k, key=silhouette_by_k.get)
print(f"Mejor k según silhouette: {best_k} (silhouette = {silhouette_by_k[best_k]:.3f})")

In [ ]:
#| label: fig-l2-kmeans
#| fig-cap: "Clusters de k-means sobre las dos primeras coordenadas del PCoA."
kmeans_labels = KMeans(n_clusters=best_k, n_init=20, random_state=42).fit_predict(pcs)
print(f"Muestras por cluster (k-means, k={best_k}): {np.bincount(kmeans_labels).tolist()}")

fig, ax = plt.subplots(figsize=(6, 5))
sns.scatterplot(x=coords["PC1"], y=coords["PC2"], hue=kmeans_labels, palette="Set2", s=70, ax=ax)
ax.set_xlabel(f"PC1 ({explained.iloc[0] * 100:.1f} %)")
ax.set_ylabel(f"PC2 ({explained.iloc[1] * 100:.1f} %)")
ax.set_title(f"k-means con k={best_k}")
ax.legend(title="Cluster")
plt.show()

La silhouette promedio esconde detalle: cada muestra tiene la suya. El **diagrama de silhouette** muestra una barra por muestra, agrupadas por cluster:

In [ ]:
#| label: fig-l2-silhouette
#| fig-cap: "Diagrama de silhouette: una barra por muestra, agrupadas por cluster."
sample_silhouette = silhouette_samples(pcs, kmeans_labels)

fig, ax = plt.subplots(figsize=(8, 5))
y_start = 0
for cluster in range(best_k):
    values = np.sort(sample_silhouette[kmeans_labels == cluster])
    ax.barh(np.arange(y_start, y_start + len(values)), values, height=1.0, label=f"cluster {cluster} (n={len(values)})")
    y_start += len(values) + 2                       # espacio entre clusters
ax.axvline(silhouette_score(pcs, kmeans_labels), color="red", linestyle="--", label="promedio")
ax.set_xlabel("Silhouette de cada muestra")
ax.set_yticks([])
ax.set_title("Diagrama de silhouette")
ax.legend(fontsize=8, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.show()

**Cuidado:** una silhouette alta **no garantiza** clusters útiles. Mira los **tamaños** de los grupos: si un cluster tiene 3 muestras y otro 67, k-means probablemente está aislando *casos atípicos*, no descubriendo grupos biológicos.

### ✏️ Ejercicio 3: Cambiar k y comparar

**Instrucción:** ejecuta k-means con **k = 6** y compara su silhouette y los tamaños de los grupos con el mejor k. ¿Cuál separaría mejor *grupos biológicos*? Tiempo estimado: 5 min.

In [ ]:
# ### TU CÓDIGO AQUÍ ###
k_exercise = TU_CODIGO_AQUI     # prueba con 6

labels_exercise = KMeans(n_clusters=k_exercise, n_init=20, random_state=42).fit_predict(pcs)
print(f"k={best_k}: silhouette = {silhouette_by_k[best_k]:.3f} | tamaños = {np.bincount(kmeans_labels).tolist()}")
print(f"k={k_exercise}: silhouette = {silhouette_score(pcs, labels_exercise):.3f} | tamaños = {np.bincount(labels_exercise).tolist()}")

<details>
<summary>Ver solución</summary>

```python
k_exercise = 6
```

Con k = 6 la silhouette baja a 0.579 (frente a 0.761 con k = 3) y los tamaños son [7, 34, 3, 3, 2, 26]. La silhouette es mayor con k = 3 **porque aísla casos extremos**; con k = 6 ya aparecen dos bloques grandes dentro del grupo principal. Una silhouette más baja puede dar una partición más informativa: hay que mirar los tamaños y el contexto biológico.
</details>

---

## 📖 Sección 5: ¿Los clusters siguen a la planta o al estudio? (~20 min)

Ahora la pregunta del taller. Tenemos grupos (clusters) calculados **sin mirar** el metadato. Comparémoslos con tres variables: la **familia de la planta**, el **país** y el **estudio**.

### 5.1 Concordancia entre clusters y etiquetas
El **índice de Rand ajustado** (ARI) mide qué tanto coinciden dos agrupaciones: **1** = idénticas, **~0** = como al azar.

In [ ]:
#| label: tbl-l2-ari
#| tbl-cap: "Índice de Rand ajustado entre los clusters y tres variables del metadato."
label_columns = {"familia de planta": "host_name_family", "país": "country", "estudio": "study_id"}
ari_table = pd.DataFrame({
    name: {"k-means": adjusted_rand_score(metadata_df[col], kmeans_labels),
           "jerárquico": adjusted_rand_score(metadata_df[col], hier_labels)}
    for name, col in label_columns.items()
}).round(3)
ari_table

### 5.2 ¿Cuánta variación explica cada variable? — PERMANOVA
Los clusters dependen de cómo cortemos. Una alternativa es preguntar directamente **cuánta de la variación entre comunidades explica cada variable**. La **PERMANOVA** hace exactamente eso sobre la matriz de distancias y da un **R²** (fracción de variación explicada) y un valor *p* (por permutaciones).

⚠ El R² crece solo por tener más grupos (19 estudios vs 3 familias), así que comparamos con el **R² ajustado**.

In [ ]:
#| label: tbl-l2-permanova
#| tbl-cap: "PERMANOVA (Bray-Curtis, 75 muestras): variación explicada por cada variable."
def permanova_summary(distance_matrix, labels, permutations=999, seed=42):
    """PERMANOVA: devuelve nº de grupos, pseudo-F, p, R² y R² ajustado."""
    labels = list(labels)
    result = permanova(distance_matrix, labels, permutations=permutations, seed=seed)
    F, p = float(result["test statistic"]), float(result["p-value"])
    n, k = len(labels), len(set(labels))
    r2 = F * (k - 1) / (F * (k - 1) + (n - k))
    return {"grupos": k, "pseudo_F": F, "p": p, "R2": r2, "R2_ajustado": 1 - (1 - r2) * (n - 1) / (n - k)}

assert list(dm.ids) == list(metadata_df.index)
permanova_table = pd.DataFrame({name: permanova_summary(dm, metadata_df[col])
                                for name, col in label_columns.items()}).T
permanova_table["grupos"] = permanova_table["grupos"].astype(int)
permanova_table.round(3)

### 5.3 Distancias entre pares: ¿qué acerca a dos muestras?
Una pregunta directa, sin clusters: dos muestras **de la misma familia de planta**, ¿están más cerca que dos de familias distintas? Y dos **del mismo estudio**, ¿están más cerca que de estudios distintos? Clasificamos **todos los pares** de muestras según lo que comparten y comparamos sus distancias:

In [ ]:
#| label: fig-l2-pares
#| fig-cap: "Distancias entre pares de muestras según si comparten estudio y/o familia de planta."
family_arr = metadata_df["host_name_family"].to_numpy()
study_arr = metadata_df["study_id"].to_numpy()
same_family = family_arr[:, None] == family_arr[None, :]
same_study = study_arr[:, None] == study_arr[None, :]

pair_type = np.select(
    [same_study & same_family, same_study & ~same_family, ~same_study & same_family],
    ["mismo estudio\nmisma familia", "mismo estudio\ndistinta familia", "distinto estudio\nmisma familia"],
    default="distinto estudio\ndistinta familia")

upper = np.triu_indices(len(dist_array), k=1)                 # cada par una sola vez
pairs_df = pd.DataFrame({"distancia": dist_array[upper], "tipo de par": pair_type[upper]})
type_order = ["mismo estudio\nmisma familia", "mismo estudio\ndistinta familia",
              "distinto estudio\nmisma familia", "distinto estudio\ndistinta familia"]

fig, ax = plt.subplots(figsize=(9, 4.5))
sns.boxplot(data=pairs_df, x="tipo de par", y="distancia", order=type_order, ax=ax)
ax.set_xlabel("")
ax.set_ylabel("Distancia de Bray-Curtis")
ax.set_title("Distancia entre pares de muestras según lo que comparten")
plt.show()

summary = pairs_df.groupby("tipo de par")["distancia"].agg(pares="size", mediana="median").loc[type_order]
summary.index = [t.replace("\n", ", ") for t in summary.index]
summary.round(3)

Compara sobre todo las **dos cajas de la derecha** (estudios distintos): ¿cambia la distancia por compartir familia? Dos advertencias: (1) los pares **no son independientes** (cada muestra aparece en muchos pares), así que no calculamos valores *p* aquí; es una comparación descriptiva; (2) hay muy pocos pares de "mismo estudio, distinta familia" (solo dos estudios incluyen dos familias).

### 5.4 ¿Quiénes son los clusters pequeños?
Revisemos las muestras que k-means separó del grupo principal:

In [ ]:
main_cluster = np.bincount(kmeans_labels).argmax()
is_outlier = kmeans_labels != main_cluster
outliers = metadata_df.loc[is_outlier, ["host_name", "study_id", "country", "seq_type"]].assign(cluster=kmeans_labels[is_outlier])
print(f"{is_outlier.sum()} muestras fuera del cluster principal:")
outliers.sort_values("cluster")

Repetimos el análisis **sin esas muestras**, para ver qué estructura hay en el grupo principal: primero el mapa (PCoA recalculado) y luego la PERMANOVA.

In [ ]:
#| label: fig-l2-pcoa-sin-atipicos
#| fig-cap: "PCoA recalculado sin las muestras atípicas: familia de planta (izq.) y estudio (der.)."
keep_ids = list(metadata_df.index[~is_outlier])
dm_main = dm.filter(keep_ids)
pcoa_main = pcoa(dm_main, number_of_dimensions=3)
explained_main = pcoa_main.proportion_explained
plot_main = pcoa_main.samples[["PC1", "PC2"]].join(metadata_df[["host_name_family", "study_id"]])

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
sns.scatterplot(data=plot_main, x="PC1", y="PC2", hue="host_name_family", s=70, ax=axes[0])
axes[0].set_title("Sin atípicos: familia de la planta")
axes[0].legend(title="Familia")
sns.scatterplot(data=plot_main, x="PC1", y="PC2", hue="study_id", palette="tab20", s=70, ax=axes[1])
axes[1].set_title("Sin atípicos: estudio")
axes[1].legend(bbox_to_anchor=(1.01, 1), loc="upper left", ncol=2, fontsize=8, title="Estudio")
for ax in axes:
    ax.set_xlabel(f"PC1 ({explained_main.iloc[0] * 100:.1f} %)")
    ax.set_ylabel(f"PC2 ({explained_main.iloc[1] * 100:.1f} %)")
plt.tight_layout()
plt.show()

In [ ]:
permanova_main = pd.DataFrame({name: permanova_summary(dm_main, metadata_df.loc[keep_ids, col])
                               for name, col in label_columns.items()}).T
permanova_main["grupos"] = permanova_main["grupos"].astype(int)
print(f"Sin atípicos: {len(keep_ids)} muestras")
permanova_main.round(3)

El mensaje principal de la lección, en una sola figura:

In [ ]:
#| label: fig-l2-r2
#| fig-cap: "R² ajustado de la PERMANOVA por variable, con todas las muestras y sin atípicos."
r2_compare = pd.DataFrame({"todas las muestras": permanova_table["R2_ajustado"],
                           "sin atípicos": permanova_main["R2_ajustado"]})

ax = r2_compare.plot(kind="bar", figsize=(6.5, 4), rot=0)
ax.set_ylabel("R² ajustado (variación explicada)")
ax.set_xlabel("")
ax.set_title("¿Cuánta variación entre comunidades explica cada variable?")
ax.legend(title="Muestras")
plt.show()

### Interpretación
Antes de leer las conclusiones, responde: **¿qué variable explica más variación? ¿Podemos afirmar que "la planta" moldea la comunidad de la rizósfera?** Recuerda lo que vimos en la Lección 1: casi cada estudio muestreó **una sola familia**. Con ese diseño, *planta* y *estudio* son prácticamente la misma cosa para los datos.

<details>
<summary>📝 Notas del instructor (cifras verificadas con el subconjunto canónico)</summary>

- **Distancias**: Bray-Curtis mediana 0.30, máxima 0.96; correlación con la euclidiana 0.91.
- **PCoA** (varianza PC1–PC5): 40.6 % · 14.9 % · 11.3 % · 7.7 % · 5.4 %. PC1 está dominado por 8 muestras aisladas.
- **Jerárquico** (enlace promedio, 3 grupos): tamaños [3, 3, 69]; cofenética 0.989 (promedio) vs 0.967 (complete).
- **k-means** (silhouette por k): k=2: 0.728 · k=3: 0.761 · k=4: 0.628 · k=5: 0.653 · k=6: 0.579 · k=7: 0.569 · k=8: 0.471. Mejor k = 3 con tamaños [67, 3, 5] → aísla casos extremos.
- **ARI** (k-means / jerárquico) frente a familia: 0.006 / 0.006 · país: 0.086 / 0.080 · estudio: 0.028 / 0.022 (todos cercanos a 0: los clusters aíslan atípicos, no grupos).
- **PERMANOVA, 75 muestras**: familia (3 grupos) R² = 0.080 (ajustado 0.054), pseudo-F = 3.12, p = 0.002; país (8) R² = 0.507 (ajustado 0.456), pseudo-F = 9.86, p = 0.001; estudio (19) R² = 0.843 (ajustado 0.793), pseudo-F = 16.72, p = 0.001.
- **Vecino más cercano** (observado / azar / veces): estudio 0.75 / 0.06 / 12.8× · país 0.88 / 0.23 / 3.8× · familia 0.89 / 0.32 / 2.8×.
- **Las 8 muestras aisladas**: 3 de frijol (E03, México, `seq_type` vacío); 4 de pimiento (3 de E15, Corea; 1 de E02, Dinamarca) y 1 de maíz (E10, tipo `PCR`, ~24 % de `Homo`). Los 4 pimientos de dos estudios y países distintos caen juntos: llamativo, pero **n = 4, no concluyente**.
- **Sin atípicos (67 muestras)**: familia R² = 0.150 (ajustado 0.123), pseudo-F = 5.65, p = 0.001; país R² = 0.414 (ajustado 0.366), pseudo-F = 8.61, p = 0.001; estudio R² = 0.762 (ajustado 0.691), pseudo-F = 10.86, p = 0.001.
- **Mensaje**: la familia es estadísticamente significativa pero explica poco, y no se puede separar del estudio porque casi cada estudio muestreó una sola familia. Los resultados de k-means pueden variar levemente con otras versiones de scikit-learn.
</details>

---

## 📊 Resumen de la lección (~5 min)

- **Bray-Curtis** compara comunidades por sus abundancias relativas; la **PCoA** la resume en un mapa.
- **Jerárquico** (dendrograma + enlace) y **k-means** (+ silhouette) agrupan sin usar el metadato. Silhouette alta no implica clusters biológicamente útiles: **mira los tamaños**.
- Al contrastar con el metadato, la **PERMANOVA** (R² ajustado) y el **vecino más cercano** dicen qué variable acompaña la estructura de los datos.
- **Asociación no es causa:** si planta y estudio están entrelazados, el análisis no puede separar sus efectos. Un buen diseño (varias plantas por estudio, varios estudios por planta) es lo que lo permitiría.

**Conexión con la Lección 3:** hemos usado proporciones (abundancias relativas) y Bray-Curtis. Los datos de microbioma son **composicionales** — sus valores suman 1 — y eso tiene consecuencias estadísticas; la siguiente lección introduce las transformaciones (CLR, ILR) para tratarlo.

---

## 💾 Exportar resultados para la siguiente lección

In [ ]:
import os
os.makedirs(RESULTS_PATH, exist_ok=True)

dist_df.to_csv(RESULTS_PATH + "leccion2_matriz_distancias_braycurtis.csv")

clusters_df = coords.copy()
clusters_df.insert(0, "kmeans_cluster", kmeans_labels)
clusters_df.insert(1, "hierarchical_cluster", hier_labels)
clusters_df.to_csv(RESULTS_PATH + "leccion2_clusters_pcoa.csv")

print(f"Exportado en: {RESULTS_PATH}")
print("  - leccion2_matriz_distancias_braycurtis.csv  (75 x 75)")
print("  - leccion2_clusters_pcoa.csv                  (clusters k-means y jerárquico + PC1..PC5)")